# 개별종목 조합F — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합F 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합F의 피처 값만 지정합니다.
import json

COMBINATION = 'F'
FEATURE_COLUMNS = (
    'ret_5_rank',
    'sector_relative_rank',
    'turnover_rank',
    'hv_20_rank',
    'market_cap_percentile',
    'sector_market_cap_rank',
    'industry_stock_rank',
    'volume_z_20',
    'bb_position',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172532 163
조합F 피처: ('ret_5_rank', 'sector_relative_rank', 'turnover_rank', 'hv_20_rank', 'market_cap_percentile', 'sector_market_cap_rank', 'industry_stock_rank', 'volume_z_20', 'bb_position')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3812,0.3701,0.0111,0.3780,0.2967,0.3473
1,2,balanced,999,20140414,20140711,0.3792,0.4743,-0.0951,0.3585,0.2591,0.3230
2,3,balanced,1248,20150421,20150716,0.3755,0.3330,0.0425,0.3725,0.2968,0.3442
3,4,balanced,1496,20160422,20160719,0.4003,0.4128,-0.0125,0.3843,0.2748,0.3433
4,5,balanced,1745,20170424,20170721,0.3719,0.4182,-0.0463,0.3520,0.2983,0.3378
5,6,balanced,1994,20180503,20180731,0.3880,0.3908,-0.0027,0.3771,0.2612,0.3312
6,7,balanced,2243,20190514,20190806,0.3799,0.4615,-0.0817,0.3537,0.1998,0.2867
7,8,balanced,2492,20200518,20200807,0.3743,0.3144,0.0599,0.3700,0.3322,0.3578
8,9,balanced,2741,20210518,20210810,0.4148,0.4423,-0.0274,0.3961,0.3905,0.4002
9,10,balanced,2989,20220519,20220812,0.3647,0.3343,0.0303,0.3572,0.2667,0.3229


,OOS 폴드 평균
accuracy,0.3814
training_majority_baseline_accuracy,0.3851
accuracy_minus_training_majority_baseline,-0.0037
macro_f1,0.3690
down_recall,0.2905
core_harmonic_mean,0.3403


재실행 명령: python scripts/run_stock_model_experiment.py
